# Pipeline de Limpieza y Preprocesamiento del Corpus

**TFM:** Evaluación experimental de Recursive Language Models para procesamiento de contexto largo en LLMs  
**Autor:** Juan Antonio Jiménez Cobo

Este notebook toma los `.txt` generados por `01_text_extraction.ipynb` y aplica
los siguientes pasos de preprocesamiento:

1. Eliminación del bloque de autores, afiliaciones y correos
2. Eliminación de la sección de referencias bibliográficas
3. Eliminación de agradecimientos y declaraciones administrativas
4. Eliminación de apéndices
5. Limpieza de marcas editoriales (watermarks, copyright, DOI inline)
6. Normalización de saltos de línea dentro de párrafos
7. Normalización de encabezados de sección
8. Gestión de fórmulas matemáticas (placeholder `[EQUATION]`)
9. Guardado como `.json` con metadatos en `corpus/processed/`
10. Generación de informe de preprocesamiento comparativo

**Estrategia de detección de secciones:** expresiones regulares flexibles.  
**Política ante secciones no detectadas:** WARNING en el informe, artículo procesado igualmente.

---

## 0. Configuración de rutas

> **IMPOTANTE:** Modifica esta celda con tus rutas correspondientes antes de ejecutar el notebook.

In [ ]:
# ============================================================
# RUTAS — ajusta según tu estructura de Google Drive
# ============================================================

# Carpeta con los .txt extraídos por el notebook 01
RAW_DIR = "/content/drive/MyDrive/corpus/raw"

# Ruta al CSV con los metadatos del corpus (exportado mediante Rayyan)
CSV_PATH = "/content/drive/MyDrive/corpus/metadata/corpus_metadata.csv"

# Ruta de salida para el informe de extracción
EXTRACTION_REPORT_PATH = "/content/drive/MyDrive/corpus/reports/extraction_report.csv"

# Carpeta de salida para los .json preprocesados
PROCESSED_DIR = "/content/drive/MyDrive/corpus/processed"

# Informe comparativo pre/post preprocesamiento
REPORT_PATH = "/content/drive/MyDrive/corpus/reports/preprocessing_report.csv"

# Número total de artículos
N_PAPERS = 55

## 1. Instalación de dependencias

In [ ]:
!pip install tiktoken --quiet

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Imports y utilidades

In [ ]:
import os
import re
import json
import pandas as pd
import tiktoken
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional

# Crear carpeta de salida si no existe
Path(PROCESSED_DIR).mkdir(parents=True, exist_ok=True)

# Tokenizador para estimar longitud en tokens (cl100k_base = GPT-4 / text-embedding-ada-002)
enc = tiktoken.get_encoding("cl100k_base")

def count_tokens(text: str) -> int:
    return len(enc.encode(text))

print("✓ Imports completados")

## 4. Cargar metadatos y informe de extracción

In [ ]:
# Metadatos del corpus
df_meta = pd.read_csv(CSV_PATH)
df_meta["paper_id"] = [f"paper_{str(i+1).zfill(2)}" for i in range(len(df_meta))]

# Informe de extracción (para comparativa pre/post)
df_extraction = pd.read_csv(EXTRACTION_REPORT_PATH)

print(f"✓ Metadatos cargados: {len(df_meta)} artículos")
print(f"✓ Informe de extracción cargado: {len(df_extraction)} filas")
df_meta.head(3)

## 5. Definición de patrones de detección

Expresiones regulares flexibles para detectar los límites de cada sección a eliminar.
Cada patrón cubre las variantes tipográficas más habituales en papers académicos:
mayúsculas, numeración, prefijos de apéndice, etc.

In [ ]:
# ------------------------------------------------------------
# Patrones de secciones a ELIMINAR
# Cada patrón busca una línea que sea (o empiece por) el
# encabezado de la sección. Se aplica sobre líneas individuales.
# ------------------------------------------------------------

PATTERNS = {
    "references": re.compile(
        r"^\s*(\d+\.?\s+)?references\s*$",
        re.IGNORECASE
    ),
    "bibliography": re.compile(
        r"^\s*(\d+\.?\s+)?bibliography\s*$",
        re.IGNORECASE
    ),
    "appendix": re.compile(
        r"^\s*(appendix|appendices|annex)(\s+[A-Z0-9][\w\.]*)?\s*[:\.\-]?\s*$",
        re.IGNORECASE
    ),
    "acknowledgments": re.compile(
        r"^\s*(\d+\.?\s+)?(acknowledgments?|acknowledgements?|funding|conflict of interest|author contributions?|data availability|ethics statement)\s*$",
        re.IGNORECASE
    ),
    "disclaimer": re.compile(
        r"^\s*Disclaimer/Publisher",
        re.IGNORECASE
    ),
}

# Patrones de marcas editoriales (se eliminan línea a línea)
EDITORIAL_PATTERNS = [
    re.compile(r"downloaded from", re.IGNORECASE),
    re.compile(r"©\s*\d{4}"),
    re.compile(r"all rights reserved", re.IGNORECASE),
    re.compile(r"doi:\s*10\.\d{4,}/\S+", re.IGNORECASE),
    re.compile(r"https?://doi\.org/\S+"),
    re.compile(r"published (by|in|online)", re.IGNORECASE),
    re.compile(r"preprint\.\s*arxiv", re.IGNORECASE),
    re.compile(r"^\s*\[CrossRef\]", re.IGNORECASE),
    re.compile(r"^\s*\[PubMed\]", re.IGNORECASE),
]

# Patrón para detectar bloques de afiliaciones al inicio
# (líneas con @ indican correos electrónicos)
EMAIL_PATTERN = re.compile(r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}")

# Patrón para detectar fórmulas matemáticas densas
# (líneas con alta densidad de símbolos matemáticos)
MATH_PATTERN = re.compile(r"[=∑∫∂∇×·⊗⊕≤≥≠≈∈∉∀∃αβγδεζηθλμπρστφψω□■▪▫◆◇○●\ufffd]{2,}")

# Patrón para detectar cabeceras/pies de página de revista
JOURNAL_HEADER_PATTERN = re.compile(
    r"^[\w\s]+\d{4},\s*\d+,\s*\d+\.?\s*(\d+\s*of\s*\d+)?\s*$"
)

# Patrón para detectar bloques de afiliaciones
AFFILIATION_PATTERN = re.compile(
    r"^\s*(Citation:|Received:|Revised:|Accepted:|Published:|Copyright:|Licensee|"
    r"This article is an open access|distributed|under|terms|conditions of the Creative)",
    re.IGNORECASE
)

print("✓ Patrones definidos")

## 6. Funciones de preprocesamiento

In [ ]:
@dataclass
class PreprocessingLog:
    """Registro de qué se eliminó y qué advertencias se generaron."""
    removed_sections: list = field(default_factory=list)
    warnings: list = field(default_factory=list)
    editorial_lines_removed: int = 0
    affiliation_lines_removed: int = 0
    math_placeholders: int = 0


def remove_author_block(lines: list, log: PreprocessingLog) -> list:
    """
    Elimina el bloque de afiliaciones, correos y columna editorial lateral
    que aparece al inicio del paper antes del abstract.
    Estrategia: busca el abstract como ancla y elimina todo lo anterior
    excepto el título.
    """
    # Buscar la línea del Abstract como ancla fiable
    abstract_idx = None
    for i, line in enumerate(lines[:80]):
        if re.match(r"^\s*Abstract\s*:", line, re.IGNORECASE):
            abstract_idx = i
            break

    if abstract_idx is None:
        log.warnings.append("No se encontró 'Abstract:' en las primeras 80 líneas — bloque de autor no eliminado")
        return lines

    # Buscar el título: la línea más larga antes del abstract
    # que no sea una marca editorial ni un email
    header_block = lines[:abstract_idx]
    title_idx = None
    for i, line in enumerate(header_block):
        stripped = line.strip()
        if (
            len(stripped) > 20
            and not EMAIL_PATTERN.search(stripped)
            and not AFFILIATION_PATTERN.match(stripped)
            and not any(p.search(stripped) for p in EDITORIAL_PATTERNS)
        ):
            title_idx = i
            break

    if title_idx is None:
        title_idx = 0

    # Conservar desde el título hasta el final
    removed = abstract_idx - title_idx
    log.removed_sections.append(
        f"author_block+editorial_sidebar (líneas 0-{abstract_idx}, {abstract_idx - title_idx} líneas eliminadas antes del título)"
    )
    log.affiliation_lines_removed = abstract_idx - title_idx

    return lines[title_idx:]


def remove_section_from(lines: list, pattern_name: str, log: PreprocessingLog) -> list:
    """
    Elimina todo el contenido desde la primera línea que coincide con
    el patrón indicado hasta el final del documento.
    Registra un WARNING si la sección no se encuentra.
    Para comprobar los encabezados de sección, comprueba que la línea anterior
    está vacía o no existe.
    """
    pattern = PATTERNS[pattern_name]
    total_lines = len(lines)

    for i, line in enumerate(lines):
        if pattern.match(line.strip()):

            # Condición 1: la línea anterior debe estar vacía
            prev_line = lines[i - 1].strip() if i > 0 else ""
            if prev_line != "":
                continue

            # Condición 2 (solo para apéndices): debe estar en la
            # segunda mitad del documento
            if pattern_name == "appendix" and i < total_lines * 0.5:
                continue

            log.removed_sections.append(
                f"{pattern_name} (desde línea {i}: '{line.strip()[:60]}')"
            )
            return lines[:i]

    log.warnings.append(f"Sección '{pattern_name}' no detectada")
    return lines


def remove_acknowledgments(lines: list, log: PreprocessingLog) -> list:
    """
    Elimina la sección de agradecimientos y declaraciones administrativas.
    A diferencia de referencias/apéndices, esta sección puede aparecer
    antes o después de las conclusiones, así que se elimina solo el bloque
    comprendido hasta la siguiente sección de nivel similar.
    """
    pattern = PATTERNS["acknowledgments"]
    # Patrón para detectar el inicio de la siguiente sección principal
    next_section = re.compile(r"^\s*(\d+\.?\s+)[A-Z][a-zA-Z\s]+$")

    result = []
    i = 0
    while i < len(lines):
        if pattern.match(lines[i].strip()):
            start = i
            i += 1
            # Avanzar hasta la siguiente sección o fin de documento
            while i < len(lines) and not next_section.match(lines[i].strip()):
                i += 1
            removed = i - start
            log.removed_sections.append(
                f"acknowledgments (líneas {start}-{i}, {removed} líneas)"
            )
        else:
            result.append(lines[i])
            i += 1

    if not any("acknowledgments" in s for s in log.removed_sections):
        log.warnings.append("Sección 'acknowledgments' no detectada")

    return result


def remove_editorial_marks(lines: list, log: PreprocessingLog) -> list:
    """
    Elimina líneas que contienen marcas editoriales: watermarks,
    avisos de copyright, DOIs inline, etc.
    """
    result = []
    removed = 0
    for line in lines:
        if any(p.search(line) for p in EDITORIAL_PATTERNS):
            removed += 1
        else:
            result.append(line)
    log.editorial_lines_removed = removed
    return result


def normalize_linebreaks(lines: list) -> list:
    """
    Reúne líneas que pertenecen al mismo párrafo pero están
    fragmentadas por la extracción de columnas del PDF.
    Heurística: una línea que no termina en punto, dos puntos,
    signo de interrogación o exclamación, y la siguiente empieza
    en minúscula, pertenecen al mismo párrafo.
    """
    result = []
    i = 0
    while i < len(lines):
        current = lines[i].rstrip()
        while (
            i + 1 < len(lines)
            and current
            and not re.search(r"[.!?:]\s*$", current)
            and lines[i + 1].strip()
            and lines[i + 1].strip()[0].islower()
        ):
            i += 1
            current = current + " " + lines[i].strip()
        result.append(current)
        i += 1
    return result


def normalize_section_headers(lines: list) -> list:
    """
    Estandariza encabezados de sección a formato 'N. Título'.
    Convierte variantes como 'II. RELATED WORK', '2 Related Work',
    'SECTION 2: Related Work' a '2. Related Work'.
    """
    # Patrón: línea que es solo un encabezado (corta, empieza por número o romano)
    header_pattern = re.compile(
        r"^\s*(\d+|[IVX]+)\.?\s+([A-Z][A-Z\s]+)\s*$"
    )
    result = []
    for line in lines:
        m = header_pattern.match(line)
        if m:
            number = m.group(1)
            title = m.group(2).strip().title()  # Title Case
            result.append(f"{number}. {title}")
        else:
            result.append(line)
    return result


def handle_math(lines: list, log: PreprocessingLog) -> list:
    """
    Sustituye líneas con alta densidad de símbolos matemáticos
    por el placeholder [EQUATION].
    Solo actúa sobre líneas donde los símbolos matemáticos
    representan más del 30% del contenido.
    """
    result = []
    placeholders = 0
    for line in lines:
        math_chars = len(MATH_PATTERN.findall(line))
        total_chars = len(line.strip())
        if total_chars > 0 and math_chars / total_chars > 0.3:
            result.append("[EQUATION]")
            placeholders += 1
        else:
            result.append(line)
    log.math_placeholders = placeholders
    return result


print("✓ Funciones definidas")

## 7. Pipeline principal de preprocesamiento

In [ ]:
def preprocess_paper(text: str) -> tuple[str, PreprocessingLog]:
    """
    Aplica el pipeline completo de preprocesamiento sobre el texto
    de un artículo. Devuelve (texto_procesado, log).

    Orden de operaciones:
    1. Eliminar bloque de autores/afiliaciones
    2. Eliminar apéndices (antes que referencias: suelen ir después)
    3. Eliminar referencias bibliográficas
    4. Eliminar agradecimientos y secciones administrativas
    5. Eliminar marcas editoriales
    6. Normalizar saltos de línea
    7. Normalizar encabezados de sección
    8. Gestionar fórmulas matemáticas
    9. Limpieza final de espacios
    """
    log = PreprocessingLog()
    lines = text.split("\n")

    # Paso 1: bloque de autores
    lines = remove_author_block(lines, log)

    # Paso 2: disclaimer
    lines = remove_section_from(lines, "disclaimer", log)

    # Paso 3: apéndices
    lines = remove_section_from(lines, "appendix", log)

    # Paso 4: referencias
    lines = remove_section_from(lines, "references", log)
    if any("references" in w for w in log.warnings):
        lines = remove_section_from(lines, "bibliography", log)

    # Paso 5: agradecimientos y secciones administrativas
    lines = remove_acknowledgments(lines, log)

    # Paso 6: marcas editoriales y cabeceras de revista
    lines = remove_editorial_marks(lines, log)
    lines = [l for l in lines if not JOURNAL_HEADER_PATTERN.match(l.strip())]

    # Paso 7: normalizar saltos de línea
    lines = normalize_linebreaks(lines)

    # Paso 8: normalizar encabezados
    lines = normalize_section_headers(lines)

    # Paso 9: fórmulas matemáticas
    lines = handle_math(lines, log)

    # Paso 10: limpieza final (junta líneas separadas y gestiona múltiples saltos
    # de línea y múltiples espacios consecutivos)
    text_out = "\n".join(lines)
    text_out = re.sub(r"\n{3,}", "\n\n", text_out)
    text_out = re.sub(r" {2,}", " ", text_out)
    text_out = text_out.strip()

    return text_out, log


print("✓ Pipeline definido")

## 8. Ejecución sobre el corpus completo

In [ ]:
report = []

for idx, row in df_meta.iterrows():
    paper_id = row["paper_id"]
    title = row.get("title", paper_id) if "title" in df_meta.columns else paper_id
    short_title = str(title)[:55] + "..." if len(str(title)) > 55 else str(title)

    txt_path = os.path.join(RAW_DIR, f"{paper_id}.txt")

    record = {
        "paper_id": paper_id,
        "title": title,
        "status": None,
        "n_tokens_raw": None,
        "n_tokens_processed": None,
        "tokens_removed": None,
        "pct_removed": None,
        "sections_removed": None,
        "warnings": None,
        "math_placeholders": None,
        "error": None,
    }

    # Recuperar tokens raw del informe de extracción
    extraction_row = df_extraction[df_extraction["paper_id"] == paper_id]
    if not extraction_row.empty:
        record["n_tokens_raw"] = int(extraction_row.iloc[0]["n_tokens"])

    if not os.path.exists(txt_path):
        record["status"] = "ERROR"
        record["error"] = ".txt no encontrado"
        print(f"  ✗ {paper_id} | .txt no encontrado")
        report.append(record)
        continue

    try:
        with open(txt_path, "r", encoding="utf-8") as f:
            raw_text = f.read()

        processed_text, log = preprocess_paper(raw_text)

        n_tokens_processed = count_tokens(processed_text)
        n_tokens_raw = record["n_tokens_raw"] or count_tokens(raw_text)
        tokens_removed = n_tokens_raw - n_tokens_processed
        pct_removed = tokens_removed / n_tokens_raw * 100 if n_tokens_raw > 0 else 0

        # Determinar status
        status = "WARNING" if log.warnings else "OK"

        # Guardar JSON
        output = {
            "paper_id": paper_id,
            "title": str(title),
            "authors": str(row.get("authors", "")) if "authors" in df_meta.columns else "",
            "year": str(row.get("year", "")) if "year" in df_meta.columns else "",
            "doi": str(row.get("doi", "")) if "doi" in df_meta.columns else "",
            "n_tokens": n_tokens_processed,
            "text": processed_text,
        }
        json_path = os.path.join(PROCESSED_DIR, f"{paper_id}.json")
        with open(json_path, "w", encoding="utf-8") as f:
            json.dump(output, f, ensure_ascii=False, indent=2)

        record.update({
            "status": status,
            "n_tokens_processed": n_tokens_processed,
            "tokens_removed": tokens_removed,
            "pct_removed": round(pct_removed, 1),
            "sections_removed": " | ".join(log.removed_sections),
            "warnings": " | ".join(log.warnings) if log.warnings else "",
            "math_placeholders": log.math_placeholders,
        })

        warn_tag = " ⚠" if log.warnings else ""
        print(f"  {'⚠' if warn_tag else '✓'} {paper_id} | "
              f"{n_tokens_raw:>7,} → {n_tokens_processed:>7,} tokens "
              f"(-{pct_removed:.1f}%){warn_tag} | {short_title}")

    except Exception as e:
        record["status"] = "ERROR"
        record["error"] = str(e)
        print(f"  ✗ {paper_id} | ERROR: {e}")

    report.append(record)

print("\n" + "="*60)
print(f"OK:      {len([r for r in report if r['status'] == 'OK'])}")
print(f"WARNING: {len([r for r in report if r['status'] == 'WARNING'])}")
print(f"ERROR:   {len([r for r in report if r['status'] == 'ERROR'])}")

## 9. Guardar informe de preprocesamiento

In [ ]:
report_df = pd.DataFrame(report)
report_df.to_csv(REPORT_PATH, index=False, encoding="utf-8")
print(f"✓ Informe guardado en: {REPORT_PATH}")
report_df

## 10. Estadísticas comparativas pre/post preprocesamiento

In [ ]:
ok = report_df[report_df["status"].isin(["OK", "WARNING"])].copy()

if len(ok) > 0:
    print("=" * 60)
    print("ESTADÍSTICAS COMPARATIVAS DEL CORPUS")
    print("=" * 60)
    print(f"  Artículos procesados:          {len(ok)}")
    print()
    print(f"  ANTES del preprocesamiento:")
    print(f"    Total tokens:                {ok['n_tokens_raw'].sum():>10,}")
    print(f"    Media tokens/artículo:       {ok['n_tokens_raw'].mean():>10,.0f}")
    print(f"    Mediana tokens/artículo:     {ok['n_tokens_raw'].median():>10,.0f}")
    print()
    print(f"  DESPUÉS del preprocesamiento:")
    print(f"    Total tokens:                {ok['n_tokens_processed'].sum():>10,}")
    print(f"    Media tokens/artículo:       {ok['n_tokens_processed'].mean():>10,.0f}")
    print(f"    Mediana tokens/artículo:     {ok['n_tokens_processed'].median():>10,.0f}")
    print()
    total_removed = ok['tokens_removed'].sum()
    pct_total = total_removed / ok['n_tokens_raw'].sum() * 100
    print(f"  REDUCCIÓN TOTAL:")
    print(f"    Tokens eliminados:           {total_removed:>10,}")
    print(f"    Reducción porcentual:        {pct_total:>9.1f}%")
    print()

    # Top 5 artículos con mayor reducción absoluta
    print("  TOP 5 ARTÍCULOS CON MAYOR REDUCCIÓN ABSOLUTA:")
    top5 = ok.nlargest(5, "tokens_removed")[["paper_id", "n_tokens_raw", "n_tokens_processed", "pct_removed"]]
    print(top5.to_string(index=False))
    print()

    # Artículos con warnings
    warnings = ok[ok["status"] == "WARNING"]
    if len(warnings) > 0:
        print(f"  ⚠ ARTÍCULOS CON WARNINGS ({len(warnings)}):")
        for _, r in warnings.iterrows():
            print(f"    - {r['paper_id']}: {r['warnings']}")
    else:
        print("  ✓ Ningún artículo con warnings")

    # Artículos sospechosos: reducción > 60% (posible sobreelminación)
    sospechosos = ok[ok["pct_removed"] > 60]
    if len(sospechosos) > 0:
        print()
        print("  ⚠ ARTÍCULOS CON REDUCCIÓN > 60% (revisar manualmente):")
        for _, r in sospechosos.iterrows():
            print(f"    - {r['paper_id']}: -{r['pct_removed']}% | {r['sections_removed'][:80]}")

## 11. Inspección manual de un artículo

Compara el texto raw con el procesado para verificar que el pipeline funciona correctamente.

In [ ]:
PAPER_TO_INSPECT = "paper_01"
N_CHARS_PREVIEW = 2000

json_path = os.path.join(PROCESSED_DIR, f"{PAPER_TO_INSPECT}.json")
txt_path  = os.path.join(RAW_DIR, f"{PAPER_TO_INSPECT}.txt")

if os.path.exists(json_path) and os.path.exists(txt_path):
    with open(txt_path, "r", encoding="utf-8") as f:
        raw = f.read()
    with open(json_path, "r", encoding="utf-8") as f:
        processed = json.load(f)

    print(f"{'='*60}")
    print(f"TEXTO RAW — primeros {N_CHARS_PREVIEW} caracteres")
    print(f"{'='*60}")
    print(raw[:N_CHARS_PREVIEW])

    print(f"\n{'='*60}")
    print(f"TEXTO PROCESADO — primeros {N_CHARS_PREVIEW} caracteres")
    print(f"{'='*60}")
    print(processed["text"][:N_CHARS_PREVIEW])

    print(f"\n{'='*60}")
    print(f"TEXTO RAW — últimos {N_CHARS_PREVIEW} caracteres")
    print(f"{'='*60}")
    print(raw[-N_CHARS_PREVIEW:])

    print(f"\n{'='*60}")
    print(f"TEXTO PROCESADO — últimos {N_CHARS_PREVIEW} caracteres")
    print(f"{'='*60}")
    print(processed["text"][-N_CHARS_PREVIEW:])
else:
    print(f"Archivos no encontrados para {PAPER_TO_INSPECT}")

In [ ]:
paper_id = "paper_14"
txt_path = os.path.join(RAW_DIR, f"{paper_id}.txt")

with open(txt_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

total_lines = len(lines)
print(f"Total líneas: {total_lines}")
print(f"Umbral 50%: línea {int(total_lines * 0.5)}")
print()

import re
APPENDIX_PATTERN = re.compile(
    r"^\s*(appendix|appendices|annex)(\s+[A-Z0-9][\w\.]*)?\s*[:\.\-]?\s*$",
    re.IGNORECASE
)

for i, line in enumerate(lines):
    if APPENDIX_PATTERN.match(line.strip()):
        prev_line = lines[i - 1].strip() if i > 0 else ""
        pct = i / total_lines * 100
        print(f"Línea {i} ({pct:.1f}% del doc): '{line.strip()}'")
        print(f"  Línea anterior: '{prev_line}'")
        print(f"  Prev vacía: {prev_line == ''}")
        print(f"  Supera umbral 50%: {i >= total_lines * 0.5}")
        print()